
# Campus Lost and Found Portal
**Assignment 3 — End-to-End Portal Design**

**Project Purpose:** This project is a console-based Campus Lost and Found Portal developed using Python. It allows students to register, log in, report lost or found items, search for available records, submit ownership claims, and review claim requests.

The portal stores information about users, items, and claims in a JSON file. It provides a simple way for students to report missing belongings and help others recover their lost items.

**How to use this notebook:**
- Run the cells from top to bottom to define all functions and perform the automated tests.
- The sample execution section demonstrates the main features using separate sample data.
- The interactive portal can be started separately by running `main()` in the final code cell.

**Note:** The sample execution and automated tests use separate data so they do not overwrite the actual portal database.



## 1. Imports, Data Structure and Persistence

This section imports the Python libraries required for the portal. The `json` library is used to store and retrieve information in JSON format, `os` is used for file and folder management, and `hashlib` is used to convert passwords into hashed values.

The portal maintains three main categories of information: users, item records, and claim requests. The `empty_data()` function creates an initial data structure containing three empty lists.

The `load_data()` function reads previously saved information from a JSON file. If the file does not exist, it returns an empty data structure. It also checks whether the file contains the required lists and handles invalid JSON data.

The `save_data()` function saves updated information to the JSON file and creates the required folder if it does not already exist.

The `make_id()` function generates unique-looking IDs for users, items, and claims based on the number of existing records. The `password_code()` function uses SHA-256 hashing to convert passwords into hashed strings before storage.


In [1]:
import json
import os
import hashlib
DATA_FILE = "data/portal_data.json"
def empty_data():
    return {"users": [], "items": [], "claims": []}
def load_data(filename=DATA_FILE):
    try:
        with open(filename, "r") as file:
            data = json.load(file)

        if not isinstance(data, dict):
            raise ValueError("Invalid data file")

        for key in ["users", "items", "claims"]:
            if key not in data or not isinstance(data[key], list):
                raise ValueError("Missing " + key)
        return data
    except FileNotFoundError:
        return empty_data()
    except json.JSONDecodeError:
        raise ValueError("Data file is damaged")
def save_data(data, filename=DATA_FILE):
    folder = os.path.dirname(filename)
    if folder != "" and not os.path.exists(folder):
        os.makedirs(folder)
    with open(filename, "w") as file:
        json.dump(data, file, indent=4)
    return True
def make_id(records, letter):
    return letter + str(len(records) + 1)
def password_code(password):
    return hashlib.sha256(password.encode()).hexdigest()


## 2. User Registration and Login

This section manages user accounts and provides basic account security.

The `register_user()` function allows new students to create accounts by entering their name, email address, and password. It removes unnecessary spaces, converts email addresses to lowercase, checks that the required information is provided, and requires passwords to contain at least six characters. It also prevents two users from registering with the same email address.

After successful validation, a new user record is created with a generated user ID. The password is stored as a SHA-256 hash instead of plain text.

The `user_login()` function verifies the entered email address and password against the stored account information. If the credentials match, it returns the user's record. Otherwise, it returns `None`.


In [2]:
def register_user(data, name, email, password):
    name = name.strip()
    email = email.strip().lower()
    if name == "" or "@" not in email or len(password) < 6:
        return None, "Please enter a valid name, email and password of at least 6 characters."
    for user in data["users"]:
        if user["email"] == email:
            return None, "Email already registered."
    user = {
        "id": make_id(data["users"], "U"),
        "name": name,
        "email": email,
        "password": password_code(password)
    }
    data["users"].append(user)
    return user, "Registration successful."
def user_login(data, email, password):
    email = email.strip().lower()
    for user in data["users"]:
        if user["email"] == email and user["password"] == password_code(password):
            return user
    return None


## 3. Lost and Found Item Records

This section allows students to report lost or found belongings and search for existing records.

The `add_record()` function collects information about an item, including its name, category, location, description, and date. It checks that the item type is either LOST or FOUND and that the required information is provided. Each valid record receives a unique item ID and an initial status of ACTIVE.

The `search_record()` function searches through the available records using keywords, item type, and location. It only displays active records and allows students to search by different criteria. Keyword and location searches are case-insensitive.

The `my_records()` function displays the items submitted by a particular user by matching the user's ID with the ID stored in each item record.


In [3]:
def add_record(data, user_id, item_type, name, category, location, description, date):
    item_type = item_type.upper()
    if item_type != "LOST" and item_type != "FOUND":
        return None, "Item type must be LOST or FOUND."
    if name.strip() == "" or category.strip() == "" or location.strip() == "":
        return None, "Name, category and location are required."
    if description.strip() == "" or date.strip() == "":
        return None, "Description and date are required."
    item = {
        "id": make_id(data["items"], "I"),
        "user_id": user_id,
        "type": item_type,
        "name": name.strip(),
        "category": category.strip(),
        "location": location.strip(),
        "description": description.strip(),
        "date": date.strip(),
        "status": "ACTIVE"
    }
    data["items"].append(item)
    return item, "Item added successfully."
def search_record(data, keyword="", item_type="", location=""):
    results = []
    for item in data["items"]:
        if item["status"] != "ACTIVE":
            continue
        if item_type != "" and item["type"] != item_type.upper():
            continue
        if location != "" and location.lower() not in item["location"].lower():
            continue
        text = item["name"] + " " + item["category"] + " " + item["description"]
        if keyword != "" and keyword.lower() not in text.lower():
            continue
        results.append(item)
    return results
def my_records(data, user_id):
    results = []
    for item in data["items"]:
        if item["user_id"] == user_id:
            results.append(item)
    return results


## 4. Claims and Resolving Lost Items

This section handles ownership claims and the process of returning lost belongings.

The `submit_claim()` function allows a student to request an item that has been reported as FOUND by another user. The student must provide proof of ownership. The function checks whether the item exists, is active, and can be claimed. It also prevents users from claiming their own items and submitting duplicate pending or approved claims.

The `my_claim_requests()` function retrieves claim requests associated with found items belonging to a particular user. This allows item owners to view the claims submitted by other students.

The `review_claim()` function allows the owner of a found item to approve or reject a pending claim. When a claim is approved, the item status changes to CLAIMED, and any other pending claims for that item are automatically rejected.

The `resolve_lost_item()` function allows a student to mark their own lost item as recovered. It checks ownership and item status before changing the status from ACTIVE to RESOLVED.


In [4]:
def submit_claim(data, user_id, item_id, proof):
    if proof.strip() == "":
        return None, "Please provide proof that the item is yours."
    selected_item = None
    for item in data["items"]:
        if item["id"] == item_id:
            selected_item = item
            break
    if selected_item is None:
        return None, "Item not found."
    if selected_item["type"] != "FOUND" or selected_item["status"] != "ACTIVE":
        return None, "This item cannot be claimed."
    if selected_item["user_id"] == user_id:
        return None, "You cannot claim your own item."
    for claim in data["claims"]:
        if claim["item_id"] == item_id and claim["user_id"] == user_id:
            if claim["status"] == "PENDING" or claim["status"] == "APPROVED":
                return None, "You have already claimed this item."
    claim = {
        "id": make_id(data["claims"], "C"),
        "item_id": item_id,
        "user_id": user_id,
        "proof": proof.strip(),
        "status": "PENDING"
    }
    data["claims"].append(claim)
    return claim, "Claim submitted successfully."
def my_claim_requests(data, owner_id):
    results = []
    for claim in data["claims"]:
        for item in data["items"]:
            if claim["item_id"] == item["id"] and item["user_id"] == owner_id:
                if item["type"] == "FOUND":
                    results.append(claim)
    return results
def review_claim(data, owner_id, claim_id, approve):
    selected_claim = None
    selected_item = None
    for claim in data["claims"]:
        if claim["id"] == claim_id:
            selected_claim = claim
            break
    if selected_claim is None:
        return False, "Claim not found."
    for item in data["items"]:
        if item["id"] == selected_claim["item_id"]:
            selected_item = item
            break
    if selected_item is None or selected_item["user_id"] != owner_id:
        return False, "You can only review claims for your own items."
    if selected_claim["status"] != "PENDING" or selected_item["status"] != "ACTIVE":
        return False, "Claim is no longer available."
    if approve:
        selected_claim["status"] = "APPROVED"
        selected_item["status"] = "CLAIMED"
        for claim in data["claims"]:
            if claim["item_id"] == selected_item["id"] and claim["status"] == "PENDING":
                claim["status"] = "REJECTED"
        return True, "Claim approved successfully."
    selected_claim["status"] = "REJECTED"
    return True, "Claim rejected."
def resolve_lost_item(data, user_id, item_id):
    for item in data["items"]:
        if item["id"] == item_id:
            if item["user_id"] != user_id or item["type"] != "LOST":
                return False, "You can only resolve your own lost item."
            if item["status"] != "ACTIVE":
                return False, "Item is already resolved."
            item["status"] = "RESOLVED"
            return True, "Item marked as resolved."
    return False, "Item not found."


## 5. Report Generation and Export

This section provides a summary of the information stored in the portal.

The `generate_report()` function calculates the total number of registered users, total items, lost items, found items, active items, closed items, and pending claims.

An item is counted as closed when its status is no longer ACTIVE, such as when it has been CLAIMED or RESOLVED.

The `export_report()` function saves the generated summary into a text file. It creates the required folder if necessary and writes the report information in a readable format.


In [5]:
def generate_report(data):
    report = {
        "users": len(data["users"]),
        "total_items": len(data["items"]),
        "lost": 0,
        "found": 0,
        "active": 0,
        "closed": 0,
        "pending_claims": 0
    }
    for item in data["items"]:
        if item["type"] == "LOST":
            report["lost"] += 1
        else:
            report["found"] += 1
        if item["status"] == "ACTIVE":
            report["active"] += 1
        else:
            report["closed"] += 1
    for claim in data["claims"]:
        if claim["status"] == "PENDING":
            report["pending_claims"] += 1
    return report
def export_report(report, filename="reports/summary.txt"):
    folder = os.path.dirname(filename)
    if folder != "" and not os.path.exists(folder):
        os.makedirs(folder)
    with open(filename, "w") as file:
        file.write("CAMPUS LOST AND FOUND REPORT\n")
        for key in report:
            file.write(key + ": " + str(report[key]) + "\n")
    return filename


## 6. Console Interface and Error Handling

This section connects the different functions of the portal through a menu-based console interface.

The `ask_number()` function accepts numerical menu choices and checks whether the entered value falls within the allowed range. It uses a `while` loop to keep asking until a valid option is entered and handles non-numeric input using `try-except`.

The `show_items()` function displays item records in a readable format. The `report_item()` function collects item details from the user and saves valid records.

The `student_menu()` function provides the main student menu after login. It allows students to report lost or found items, search records, view their reports, submit claims, review claims, resolve lost items, view summaries, export reports, and log out.

The `main()` function starts the portal and provides the registration, login, and exit options. It also handles file-related errors, unexpected program interruptions, and invalid input.

The `while` loops keep the menus running until the user logs out or exits. The `for` loops are used to search through and display records. The `finally` block ensures that a closing message is displayed when the program ends.


In [6]:
def ask_number(message, minimum, maximum):
    while True:
        try:
            number = int(input(message))
            if number >= minimum and number <= maximum:
                return number
            print("Please select a valid option.")
        except ValueError:
            print("Please enter a number.")
def show_items(items):
    if len(items) == 0:
        print("No items found.")
    for item in items:
        print("\nID:", item["id"], "| Type:", item["type"], "| Status:", item["status"])
        print("Name:", item["name"], "| Category:", item["category"])
        print("Location:", item["location"], "| Date:", item["date"])
        print("Description:", item["description"])
def report_item(data, user, item_type):
    name = input("Item name: ")
    category = input("Category: ")
    location = input("Location: ")
    description = input("Description: ")
    date = input("Date (YYYY-MM-DD): ")
    item, message = add_record(data, user["id"], item_type, name, category,
                               location, description, date)
    print(message)
    if item is not None:
        save_data(data)
        print("Item ID:", item["id"])
def student_menu(data, user):
    while True:
        print("\n--- CAMPUS LOST AND FOUND ---")
        print("Welcome,", user["name"])
        print("1. Report Lost Item")
        print("2. Report Found Item")
        print("3. Search Items")
        print("4. My Reports")
        print("5. Claim Found Item")
        print("6. View and Review Claims")
        print("7. Mark Lost Item as Recovered")
        print("8. View Summary")
        print("9. Export Report")
        print("10. Logout")
        choice = ask_number("Enter choice: ", 1, 10)
        if choice == 1:
            report_item(data, user, "LOST")
        elif choice == 2:
            report_item(data, user, "FOUND")
        elif choice == 3:
            keyword = input("Search keyword: ").strip()
            item_type = input("Type (LOST/FOUND): ").strip()
            location = input("Location: ").strip()
            results = search_record(data, keyword, item_type, location)
            show_items(results)
        elif choice == 4:
            results = my_records(data, user["id"])
            show_items(results)
        elif choice == 5:
            results = search_record(data, "", "FOUND", "")
            show_items(results)
            item_id = input("Enter item ID: ").strip().upper()
            proof = input("Proof of ownership: ")
            claim, message = submit_claim(data, user["id"], item_id, proof)
            print(message)
            if claim is not None:
                save_data(data)
                print("Claim ID:", claim["id"])
        elif choice == 6:
            claims = my_claim_requests(data, user["id"])
            if len(claims) == 0:
                print("No claims found.")
            for claim in claims:
                print("\nClaim ID:", claim["id"])
                print("Item ID:", claim["item_id"])
                print("Status:", claim["status"])
                print("Proof:", claim["proof"])
            if len(claims) > 0:
                answer = input("Review a claim? (y/n): ").strip().lower()
                if answer == "y":
                    claim_id = input("Enter claim ID: ").strip().upper()
                    decision = input("Approve claim? (y/n): ").strip().lower()
                    if decision == "y" or decision == "n":
                        changed, message = review_claim(
                            data, user["id"], claim_id, decision == "y"
                        )
                        print(message)
                        if changed:
                            save_data(data)
                    else:
                        print("Invalid choice.")
        elif choice == 7:
            results = my_records(data, user["id"])
            show_items(results)
            item_id = input("Enter lost item ID: ").strip().upper()
            changed, message = resolve_lost_item(data, user["id"], item_id)
            print(message)
            if changed:
                save_data(data)
        elif choice == 8:
            report = generate_report(data)
            for key in report:
                print(key, ":", report[key])
        elif choice == 9:
            report = generate_report(data)
            filename = export_report(report)
            print("Report saved at:", filename)
        else:
            print("Logged out successfully.")
            break
def main():
    try:
        data = load_data()
    except (ValueError, PermissionError, OSError) as error:
        print("Error loading data:", error)
        return
    try:
        while True:
            print("\n=== CAMPUS LOST AND FOUND PORTAL ===")
            print("1. Register")
            print("2. Login")
            print("3. Exit")
            choice = ask_number("Enter choice: ", 1, 3)
            try:
                if choice == 1:
                    name = input("Name: ")
                    email = input("Email: ")
                    password = input("Password: ")
                    user, message = register_user(data, name, email, password)
                    print(message)
                    if user is not None:
                        save_data(data)
                elif choice == 2:
                    email = input("Email: ")
                    password = input("Password: ")
                    user = user_login(data, email, password)
                    if user is None:
                        print("Invalid email or password.")
                    else:
                        student_menu(data, user)
                else:
                    print("Exiting portal.")
                    break
            except (PermissionError, OSError) as error:
                print("File error:", error)
    except (KeyboardInterrupt, EOFError):
        print("\nProgram stopped.")
    finally:
        print("Thank you for using the portal.")


## 7. Automated Functional Tests

This section tests the main functions of the Campus Lost and Found Portal to check whether they work correctly.

The `run_tests()` function uses Python's `assert` statements to verify different operations, including user registration, successful and unsuccessful login attempts, duplicate email prevention, adding items, searching records, submitting claims, approving claims, resolving lost items, and generating reports.

It also tests file handling by saving and loading JSON data, checking the behavior when a file is missing, and handling invalid JSON content.

A temporary directory is used for file testing so that the original portal database is not affected. If all the conditions are satisfied, the program displays the message **"All tests passed successfully!"**.


In [7]:
import tempfile
def run_tests():
    data = empty_data()
    user1, msg = register_user(data, "Ali", "ali@campus.edu", "secret123")
    assert user1 is not None
    assert user_login(data, "ali@campus.edu", "secret123") is not None
    assert user_login(data, "ali@campus.edu", "wrongpass") is None
    duplicate, msg = register_user(data, "Other", "ali@campus.edu", "secret123")
    assert duplicate is None
    user2, msg = register_user(data, "Sara", "sara@campus.edu", "secret123")
    found, msg = add_record(data, user1["id"], "FOUND", "Black USB", "Electronics", "Library", "Blue cap", "2026-11-09")
    lost, msg = add_record(data, user2["id"], "LOST", "Wallet", "Accessories", "Cafeteria", "Brown wallet", "2026-11-09")
    assert found is not None
    assert lost is not None
    assert len(search_record(data, "USB")) == 1
    assert len(search_record(data, "", "LOST")) == 1
    assert len(my_records(data, user1["id"])) == 1
    claim, msg = submit_claim(data, user2["id"], found["id"], "The cap is blue")
    assert claim is not None
    bad_claim, msg = submit_claim(data, user1["id"], found["id"], "Mine")
    assert bad_claim is None
    denied, msg = review_claim(data, user2["id"], claim["id"], True)
    assert denied == False
    approved, msg = review_claim(data, user1["id"], claim["id"], True)
    assert approved == True
    assert found["status"] == "CLAIMED"
    resolved, msg = resolve_lost_item(data, user2["id"], lost["id"])
    assert resolved == True
    assert lost["status"] == "RESOLVED"
    report = generate_report(data)
    assert report["users"] == 2
    assert report["total_items"] == 2
    assert report["closed"] == 2
    with tempfile.TemporaryDirectory() as folder:
        filepath = os.path.join(folder, "test.json")
        save_data(data, filepath)
        restored = load_data(filepath)
        assert restored == data
        report_file = export_report(report, os.path.join(folder, "summary.txt"))
        assert os.path.exists(report_file)
        missing_file = os.path.join(folder, "missing.json")
        assert load_data(missing_file) == empty_data()
        with open(filepath, "w") as file:
            file.write("not valid json")
        try:
            load_data(filepath)
            assert False
        except ValueError:
            pass
    print("All tests passed successfully!")
run_tests()

All tests passed successfully!



## 8. Sample Execution and Demonstration

This section demonstrates how the main features of the portal work using sample users and sample item records.

Two sample students are registered, and their login details are verified. One student reports a found item, while the other reports a lost item. The second student then searches for the found item and submits a claim by providing proof of ownership.

The owner of the found item views the claim and approves it. The item status is updated to CLAIMED. The second student also marks their own lost item as recovered, changing its status to RESOLVED.

Finally, the portal generates a summary report and exports it into a text file.

This demonstration uses a separate in-memory data structure and temporary files, keeping the actual portal database unchanged.


In [8]:

demo_data = empty_data()
print("CAMPUS LOST AND FOUND PORTAL")
user1, msg = register_user(demo_data, "Ali", "ali@gmail.com", "123456")
print(msg)
user2, msg = register_user(demo_data, "Sara", "sara@gmail.com", "123456")
print(msg)
print("Login:")
user = user_login(demo_data, "ali@gmail.com", "123456")
print("Login successful!" if user else "Login failed!")
print("Adding items:")
found, msg = add_record(demo_data, user1["id"], "FOUND", "Black USB", "Electronics", "Library", "USB with blue cap", "2026-11-10")
print(msg)
lost, msg = add_record(demo_data, user2["id"], "LOST", "Brown Wallet", "Accessories", "Cafeteria", "Brown leather wallet", "2026-11-11")
print(msg)
print("Searching for an item:")
results = search_record(demo_data, "USB")
for item in results:
    print(item["name"], item["location"], item["status"])
print("Submitting a claim:")
claim, msg = submit_claim(demo_data, user2["id"], found["id"], "It has my blue cap.")
print(msg)
print("Reviewing the claim:")
approved, msg = review_claim(demo_data, user1["id"], claim["id"], True)
print(msg)
print("Claim status:", claim["status"])
print("Item status:", found["status"])
print("Recovering a lost item:")
resolved, msg = resolve_lost_item(demo_data, user2["id"], lost["id"])
print(msg)
print("Lost item status:", lost["status"])
print("Generating report:")
report = generate_report(demo_data)
for key, value in report.items():
    print(key, ":", value)
print("Saving and exporting data:")
with tempfile.TemporaryDirectory() as folder:
    save_data(demo_data, os.path.join(folder, "data.json"))
    loaded = load_data(os.path.join(folder, "data.json"))
    print("Data saved and loaded successfully!")
    export_report(report, os.path.join(folder, "report.txt"))
    print("Report exported successfully!")
print("Sample execution completed!")


CAMPUS LOST AND FOUND PORTAL
Registration successful.
Registration successful.
Login:
Login successful!
Adding items:
Item added successfully.
Item added successfully.
Searching for an item:
Black USB Library ACTIVE
Submitting a claim:
Claim submitted successfully.
Reviewing the claim:
Claim approved successfully.
Claim status: APPROVED
Item status: CLAIMED
Recovering a lost item:
Item marked as resolved.
Lost item status: RESOLVED
Generating report:
users : 2
total_items : 2
lost : 1
found : 1
active : 0
closed : 2
pending_claims : 0
Saving and exporting data:
Data saved and loaded successfully!
Report exported successfully!
Sample execution completed!



## 9. Start the Interactive Portal

The following section provides access to the complete interactive Campus Lost and Found Portal.

After running all the previous cells, the portal can be started by uncommenting the `main()` function call in the code cell below.

The user can then interact with the menu by entering the required information and selecting different options. The portal saves valid changes in the JSON database file located at `data/portal_data.json`.

Run this cell only when you want to use the interactive portal. It is not necessary to run it when performing the sample demonstration or automated tests.


In [16]:
main()


=== CAMPUS LOST AND FOUND PORTAL ===
1. Register
2. Login
3. Exit


Enter choice:  2
Email:  sara@lhr.nu.edu.pk
Password:  11223344



--- CAMPUS LOST AND FOUND ---
Welcome, sara
1. Report Lost Item
2. Report Found Item
3. Search Items
4. My Reports
5. Claim Found Item
6. View and Review Claims
7. Mark Lost Item as Recovered
8. View Summary
9. Export Report
10. Logout


Enter choice:  2
Item name:  USB
Category:  electronics
Location:  library
Description:  black USB with white cap
Date (YYYY-MM-DD):  2026-11-8


Item added successfully.
Item ID: I5

--- CAMPUS LOST AND FOUND ---
Welcome, sara
1. Report Lost Item
2. Report Found Item
3. Search Items
4. My Reports
5. Claim Found Item
6. View and Review Claims
7. Mark Lost Item as Recovered
8. View Summary
9. Export Report
10. Logout


Enter choice:  10


... (23 outputs left out)

Enter choice:  10


Logged out successfully.

=== CAMPUS LOST AND FOUND PORTAL ===
1. Register
2. Login
3. Exit


Enter choice:  3


Exiting portal.
Thank you for using the portal.



## 10. Project Workflow and Conclusion

The Campus Lost and Found Portal provides a complete process for reporting and recovering lost belongings within a university.

The general workflow begins when students register and log in to their accounts. They can then report lost or found items, search through active records, and view their own submissions.

When a student recognizes an item that belongs to them, they can submit a claim with proof of ownership. The person who reported the found item can review the request and either approve or reject it. Once approved, the item is marked as CLAIMED. Similarly, students can mark their own lost items as RESOLVED when they have been recovered.

The portal also generates summary reports to provide an overview of users, item records, and pending claims.

The project demonstrates the practical use of Python functions, conditional statements, loops, data structures, file handling, exception handling, JSON storage, and password hashing in developing a simple information management system.
